In [ ]:
# Cell 1 — Mount Drive and check the Colab runtime (CPU is sufficient for Phase 0B).
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import subprocess
import sys

drive.mount("/content/drive")

DRIVE_BASE = Path("/content/drive/MyDrive/ProofLens")
SOURCE_ARCHIVE = DRIVE_BASE / "prooflens_code.zip"
WORKSPACE = Path("/content/prooflens_workspace")
DATA_DIR = Path("/content/datasets/tiny_genimage")
OUTPUT_DIR = Path("/content/prooflens_phase0b")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
DRIVE_OUTPUT = DRIVE_BASE / "phase0b_runs" / RUN_ID

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
free_gib = shutil.disk_usage("/content").free / 2**30
print(f"Python: {sys.version.split()[0]}")
print(f"Free Colab disk: {free_gib:.1f} GiB")
print(f"Source archive present: {SOURCE_ARCHIVE.is_file()}")
print(f"Phase 0B run: {RUN_ID}")
if free_gib < 18:
    raise RuntimeError("Less than 18 GiB free. Check storage before downloading the dataset.")

In [ ]:
# Cell 2 — Load the team's existing ProofLens source archive, without rebuilding it.
from zipfile import ZipFile

if not SOURCE_ARCHIVE.is_file():
    raise FileNotFoundError(
        f"Missing {SOURCE_ARCHIVE}. Run 'python scripts/package.py' locally "
        "and upload artifacts/prooflens_code.zip to MyDrive/ProofLens/."
    )

WORKSPACE.mkdir(parents=True, exist_ok=True)
with ZipFile(SOURCE_ARCHIVE) as archive:
    for member in archive.infolist():
        destination = (WORKSPACE / member.filename).resolve()
        if destination != WORKSPACE.resolve() and WORKSPACE.resolve() not in destination.parents:
            raise ValueError(f"Unsafe ZIP entry: {member.filename}")
    archive.extractall(WORKSPACE)

candidates = [
    p.parent
    for p in WORKSPACE.rglob("pyproject.toml")
    if (p.parent / "scripts" / "audit.py").is_file()
]
if len(candidates) != 1:
    raise RuntimeError(
        f"Expected one ProofLens project root; found {len(candidates)}: {candidates}"
    )
PROJECT_ROOT = candidates[0]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "kagglehub",
        "pandas",
        "Pillow",
        "PyYAML",
        "tqdm",
    ],
    check=True,
)
# Preserve Colab's existing CUDA-enabled PyTorch environment.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT), "--no-deps"], check=True
)
os.chdir(PROJECT_ROOT)

import prooflens

print("ProofLens package:", Path(prooflens.__file__).resolve())
print("Project root:", PROJECT_ROOT)
print("Source ZIP SHA-256:", hashlib.sha256(SOURCE_ARCHIVE.read_bytes()).hexdigest())

In [ ]:
# Cell 3 — Download Tiny GenImage directly to Colab (never to local Windows).
import kagglehub

DATA_DIR.mkdir(parents=True, exist_ok=True)
DATASET_ID = "yangsangtai/tiny-genimage"

# KaggleHub can use the KAGGLE_API_TOKEN Colab secret if authentication is required.
# Add it in Colab's Secrets panel; never paste the token into notebook cells.
DATASET_ROOT = Path(kagglehub.dataset_download(DATASET_ID, output_dir=str(DATA_DIR))).resolve()
if not DATASET_ROOT.is_dir():
    raise RuntimeError(f"Dataset directory not found: {DATASET_ROOT}")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
image_count = 0
extension_counts = {}
for current_root, _, names in os.walk(DATASET_ROOT):
    for name in names:
        extension = Path(name).suffix.lower()
        if extension in IMAGE_EXTENSIONS:
            image_count += 1
            extension_counts[extension] = extension_counts.get(extension, 0) + 1

print("Dataset root:", DATASET_ROOT)
print("Images discovered by extension:", image_count)
print("Extensions:", dict(sorted(extension_counts.items())))
print("Disk free after download:", round(shutil.disk_usage("/content").free / 2**30, 1), "GiB")
if image_count == 0:
    raise RuntimeError("No images discovered. Inspect the downloaded directory before auditing.")

In [ ]:
# Cell 4 — Run the existing repository audit implementation, not a notebook rewrite.
MANIFEST_PATH = OUTPUT_DIR / "images.csv"
SUMMARY_PATH = OUTPUT_DIR / "dataset_summary.json"

command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "audit.py"),
    "--root",
    str(DATASET_ROOT),
    "--manifest-out",
    str(MANIFEST_PATH),
    "--summary-out",
    str(SUMMARY_PATH),
]
print("Running:", " ".join(command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

if not MANIFEST_PATH.is_file() or not SUMMARY_PATH.is_file():
    raise RuntimeError("Audit did not produce the expected CSV and JSON files.")

import pandas as pd

manifest = pd.read_csv(MANIFEST_PATH)
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
print("Manifest rows:", len(manifest))
print("Columns:", list(manifest.columns))
print("Summary keys:", list(summary.keys()) if isinstance(summary, dict) else type(summary))
print("Generator distribution:")
print(manifest["generator"].value_counts(dropna=False).to_string())
print("Label distribution:")
print(manifest["label"].value_counts(dropna=False).to_string())

In [ ]:
# Cell 5 — Validate the audit, write a review, and export ONLY small artifacts to Drive.
import pandas as pd

required = {"image_id", "relative_path", "generator", "label", "source_split"}
missing = sorted(required - set(manifest.columns))
invalid_labels = (
    int((~pd.to_numeric(manifest["label"], errors="coerce").isin([0, 1])).sum())
    if not missing
    else -1
)
unknown_generators = (
    int(
        (
            manifest["generator"].isna()
            | manifest["generator"]
            .astype(str)
            .str.lower()
            .isin({"unknown", "unmapped", "n/a", "none", ""})
        ).sum()
    )
    if not missing
    else -1
)
missing_splits = int(manifest["source_split"].isna().sum()) if not missing else -1
id_duplicates = int(manifest["image_id"].duplicated().sum()) if not missing else -1
image_count_matches = len(manifest) == image_count

cross_split_duplicates = None
exact_duplicates = None
hash_coverage = 0
if "sha256" in manifest.columns:
    hashes = manifest.dropna(subset=["sha256"]).copy()
    hashes = hashes[hashes["sha256"].astype(str).str.len() == 64]
    hash_coverage = len(hashes)
    exact_duplicates = int(hashes.duplicated("sha256").sum())
    cross_split_duplicates = int((hashes.groupby("sha256")["source_split"].nunique() > 1).sum())

valid = (
    not missing
    and invalid_labels == 0
    and unknown_generators == 0
    and missing_splits == 0
    and id_duplicates == 0
    and image_count_matches
    and hash_coverage == len(manifest)
)
if cross_split_duplicates is not None and cross_split_duplicates > 0:
    valid = False
status = "AUDIT PASSED BASIC CHECKS" if valid else "REQUIRES REVIEW"

by_generator = manifest.groupby(["generator", "label"], dropna=False).size().unstack(fill_value=0)
by_split = manifest.groupby(["source_split", "label"], dropna=False).size().unstack(fill_value=0)
format_table = (
    manifest.groupby(["label", "image_format"], dropna=False).size().unstack(fill_value=0)
    if "image_format" in manifest.columns
    else None
)
format_text = format_table.to_string() if format_table is not None else "Not reported by audit"
decode_text = (
    manifest["decode_status"].value_counts(dropna=False).to_string()
    if "decode_status" in manifest.columns
    else "Not reported by audit"
)

report = f"""# ProofLens Phase 0B — Dataset Audit Review

- Run ID: `{RUN_ID}`
- Dataset: `{DATASET_ID}`
- Source archive SHA-256: `{hashlib.sha256(SOURCE_ARCHIVE.read_bytes()).hexdigest()}`
- Status: **{status}**
- Images found by file extension: **{image_count}**
- Manifest rows: **{len(manifest)}**
- Count match: **{image_count_matches}**
- Invalid/missing binary labels: **{invalid_labels}**
- Unknown generator values: **{unknown_generators}**
- Missing source splits: **{missing_splits}**
- Rows with valid SHA-256 hashes: **{hash_coverage}**
- Duplicate image IDs: **{id_duplicates}**
- Exact duplicate files (additional rows): **{exact_duplicates}**
- Hashes appearing in multiple source splits: **{cross_split_duplicates}**

## Counts by generator and label

```text
{by_generator.to_string()}
```

## Counts by source split and label

```text
{by_split.to_string()}
```

## File formats by class

```text
{format_text}
```

## Image decode status

```text
{decode_text}
```

## Limitations

- Exact SHA-256 duplicates are not the same as near-duplicates.
- Source split names and generator mappings must be checked against actual data provenance.
- Original dataset contents were not copied to Drive.
- This phase did not train or evaluate a CNN.
"""
REVIEW_PATH = OUTPUT_DIR / "phase0b_review.md"
REVIEW_PATH.write_text(report, encoding="utf-8")
metadata = {
    "run_id": RUN_ID,
    "dataset_id": DATASET_ID,
    "dataset_root": str(DATASET_ROOT),
    "source_zip_sha256": hashlib.sha256(SOURCE_ARCHIVE.read_bytes()).hexdigest(),
    "status": status,
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
for source in [MANIFEST_PATH, SUMMARY_PATH, REVIEW_PATH, OUTPUT_DIR / "run_metadata.json"]:
    shutil.copy2(source, DRIVE_OUTPUT / source.name)

print(report)
print("Saved reports to:", DRIVE_OUTPUT)
if not valid:
    print("ACTION REQUIRED: Resolve audit discrepancies before training or making research claims.")